# 04 — Select Border County Pairs

This notebook identifies contiguous cross-state county pairs where the two states had **different
minimum wages in at least one year** between 2010 and 2024. These pairs form the core of the
border-county difference-in-differences design (Dube, Lester & Reich 2010).

**Inputs:**
- `data/raw/county_adjacency.txt` — Census Bureau county adjacency file (downloaded here if missing)
- `data/intermediate/min_wage_panel.parquet` — cleaned annual state minimum wages

**Processing steps:**
1. Download and parse the Census county adjacency file.
2. Keep only **cross-state** pairs (state FIPS of county 1 ≠ state FIPS of county 2).
3. Merge minimum wages for both states in each pair, for each year 2010–2024.
4. Keep only pairs where the two states had **different minimum wages in at least one year** — these are the pairs with treatment variation.
5. Assign a unique `pair_id` and save.

**Output:** `data/intermediate/border_pairs.parquet`
- Columns: `pair_id`, `fips1`, `fips2`, `state1`, `state2`

In [ ]:
import io
import requests
import pandas as pd
from pathlib import Path

# Paths
ROOT = Path().resolve().parent
RAW = ROOT / "data" / "raw"
INTERMEDIATE = ROOT / "data" / "intermediate"
INTERMEDIATE.mkdir(parents=True, exist_ok=True)

ADJACENCY_FILE = RAW / "county_adjacency.txt"
MIN_WAGE_FILE = INTERMEDIATE / "min_wage_panel.parquet"
OUTPUT = INTERMEDIATE / "border_pairs.parquet"

print("Root            :", ROOT)
print("Adjacency file  :", ADJACENCY_FILE)
print("Min wage panel  :", MIN_WAGE_FILE)
print("Output          :", OUTPUT)

Root            : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final
Adjacency file  : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/raw/county_adjacency.txt
Min wage panel  : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/intermediate/min_wage_panel.parquet
Output          : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/intermediate/border_pairs.parquet


## 1. Download the Census county adjacency file

In [ ]:
ADJACENCY_URL = "https://www2.census.gov/geo/docs/reference/county_adjacency.txt"

if ADJACENCY_FILE.exists():
    print(f"[skip] {ADJACENCY_FILE.name} already exists")
else:
    print(f"[download] {ADJACENCY_URL}")
    resp = requests.get(ADJACENCY_URL, timeout=60)
    resp.raise_for_status()
    ADJACENCY_FILE.write_bytes(resp.content)
    print(
        f"[done] {ADJACENCY_FILE.name} ({ADJACENCY_FILE.stat().st_size / 1e3:.1f} KB)"
    )

[download] https://www2.census.gov/geo/docs/reference/county_adjacency.txt
[done] county_adjacency.txt (726.7 KB)


## 2. Parse the adjacency file

The Census adjacency file is a tab-delimited text file. Each block starts with an **anchor county**
(name + FIPS on the first line), followed by indented lines listing every county adjacent to it
(including itself). Format:

```
"Autauga County, AL"\t01001\t"Autauga County, AL"\t01001
\t\t"Chilton County, AL"\t01021
\t\t"Dallas County, AL"\t01047
```

We parse this into a flat table of (fips1, fips2) pairs and keep only **cross-state** pairs.

In [3]:
def parse_adjacency(path: Path) -> pd.DataFrame:
    """Parse Census county_adjacency.txt into a DataFrame of (fips1, fips2) pairs."""
    pairs = []
    current_fips = None

    with open(path, encoding="latin-1") as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            # Lines with an anchor county start with a non-empty county name
            if parts[0].strip():
                # parts[1] is the anchor county FIPS
                current_fips = parts[1].strip().zfill(5)
                neighbor_fips = parts[3].strip().zfill(5)
            else:
                # Continuation line — neighbor only
                neighbor_fips = parts[3].strip().zfill(5)

            if current_fips and neighbor_fips and current_fips != neighbor_fips:
                pairs.append((current_fips, neighbor_fips))

    df = pd.DataFrame(pairs, columns=["fips1", "fips2"])
    return df


adj_raw = parse_adjacency(ADJACENCY_FILE)
print(f"Total directed pairs parsed : {len(adj_raw):,}")
print(adj_raw.head(10))

Total directed pairs parsed : 18,967
   fips1  fips2
0  01001  01021
1  01001  01047
2  01001  01051
3  01001  01085
4  01001  01101
5  01003  01025
6  01003  01053
7  01003  01097
8  01003  01099
9  01003  01129


In [ ]:
# Keep only cross-state pairs (state FIPS = first 2 digits of county FIPS)
adj_raw["state1"] = adj_raw["fips1"].str[:2]
adj_raw["state2"] = adj_raw["fips2"].str[:2]

cross_state = adj_raw[adj_raw["state1"] != adj_raw["state2"]].copy()
print(f"Cross-state directed pairs  : {len(cross_state):,}")

# Deduplicate: keep canonical form where fips1 < fips2 so each pair appears once
cross_state["fips1"], cross_state["fips2"] = (
    cross_state[["fips1", "fips2"]].min(axis=1),
    cross_state[["fips1", "fips2"]].max(axis=1),
)
cross_state["state1"] = cross_state["fips1"].str[:2]
cross_state["state2"] = cross_state["fips2"].str[:2]

cross_state = cross_state.drop_duplicates(subset=["fips1", "fips2"]).reset_index(
    drop=True
)
print(f"Unique cross-state pairs    : {len(cross_state):,}")
print(
    f"Unique states represented   : {pd.concat([cross_state['state1'], cross_state['state2']]).nunique()}"
)
cross_state.head(10)

Cross-state directed pairs  : 2,616
Unique cross-state pairs    : 1,308
Unique states represented   : 49


,fips1,fips2,state1,state2
0,01003,12033,01,12
1,01005,13061,01,13
2,01005,13239,01,13
3,01005,13259,01,13
4,01017,13145,01,13
5,01017,13285,01,13
6,01019,13055,01,13
7,01019,13115,01,13
8,01019,13233,01,13
9,01023,28023,01,28


## 3. Map state FIPS → state abbreviation

We need state abbreviations to merge with the minimum wage panel (which uses 2-letter abbr).

In [ ]:
# Load the Census FIPS table (downloaded by src/01_download_data.py)
state_fips_df = pd.read_csv(
    ROOT / "data" / "raw" / "state_fips.txt",
    sep="|",
    dtype=str,
)[["STATE", "STUSAB"]].rename(columns={"STATE": "state_fips", "STUSAB": "state_abbr"})
state_fips_df["state_fips"] = state_fips_df["state_fips"].str.zfill(2)

# Build a fips → abbreviation dict
fips_to_abbr = dict(zip(state_fips_df["state_fips"], state_fips_df["state_abbr"]))

# Map onto the cross-state pairs
cross_state["state1_abbr"] = cross_state["state1"].map(fips_to_abbr)
cross_state["state2_abbr"] = cross_state["state2"].map(fips_to_abbr)

# Check for any unmapped states (territories not in min wage data)
unmapped = cross_state[
    cross_state["state1_abbr"].isna() | cross_state["state2_abbr"].isna()
]
print(f"Pairs with unmapped state FIPS: {len(unmapped)}")
if not unmapped.empty:
    print(unmapped[["state1", "state2"]].drop_duplicates())

cross_state.head(5)

Pairs with unmapped state FIPS: 0


,fips1,fips2,state1,state2,state1_abbr,state2_abbr
0,01003,12033,01,12,AL,FL
1,01005,13061,01,13,AL,GA
2,01005,13239,01,13,AL,GA
3,01005,13259,01,13,AL,GA
4,01017,13145,01,13,AL,GA


## 4. Filter to pairs with minimum wage divergence

Load the minimum wage panel and check, for each pair, whether the two states ever had **different
minimum wages** in any year 2010–2024. Pairs where both states always had the same wage (e.g. both
always at the federal floor) provide no treatment variation and are dropped.

In [6]:
# Load minimum wage panel
mw = pd.read_parquet(MIN_WAGE_FILE)
print("Min wage panel shape:", mw.shape)
mw.head()

Min wage panel shape: (814, 4)


,state,state_fips,year,min_wage
0,AK,02,2009,7.15
1,AK,02,2010,7.75
2,AK,02,2011,7.75
3,AK,02,2012,7.75
4,AK,02,2013,7.75


In [ ]:
# Filter to analysis years only
FIRST_YEAR = 2010
LAST_YEAR = 2024
mw_analysis = mw[(mw["year"] >= FIRST_YEAR) & (mw["year"] <= LAST_YEAR)]

# Merge minimum wage for state1
pairs = cross_state.merge(
    mw_analysis[["state", "year", "min_wage"]].rename(
        columns={"state": "state1_abbr", "min_wage": "mw1"}
    ),
    on="state1_abbr",
    how="inner",
)

# Merge minimum wage for state2
pairs = pairs.merge(
    mw_analysis[["state", "year", "min_wage"]].rename(
        columns={"state": "state2_abbr", "min_wage": "mw2"}
    ),
    on=["state2_abbr", "year"],
    how="inner",
)

print(f"Pair×year rows after merging min wages: {len(pairs):,}")
pairs.head()

Pair×year rows after merging min wages: 19,492


,fips1,fips2,state1,state2,state1_abbr,state2_abbr,year,mw1,mw2
0,01003,12033,01,12,AL,FL,2010,7.25,7.25
1,01003,12033,01,12,AL,FL,2011,7.25,7.25
2,01003,12033,01,12,AL,FL,2012,7.25,7.67
3,01003,12033,01,12,AL,FL,2013,7.25,7.79
4,01003,12033,01,12,AL,FL,2014,7.25,7.93


In [ ]:
# For each pair, flag years where the two states had different minimum wages
pairs["mw_differs"] = pairs["mw1"] != pairs["mw2"]

# Keep only pairs where the wages diverged in at least one year
diverging_pairs = (
    pairs.groupby(["fips1", "fips2", "state1", "state2", "state1_abbr", "state2_abbr"])[
        "mw_differs"
    ]
    .any()
    .reset_index()
)
diverging_pairs = diverging_pairs[diverging_pairs["mw_differs"]].drop(
    columns="mw_differs"
)

print(f"Total cross-state pairs       : {len(cross_state):,}")
print(f"Pairs with MW divergence      : {len(diverging_pairs):,}")
print(f"Pairs dropped (always same MW): {len(cross_state) - len(diverging_pairs):,}")
print(
    f"Unique states in treated pairs: {pd.concat([diverging_pairs['state1_abbr'], diverging_pairs['state2_abbr']]).nunique()}"
)

Total cross-state pairs       : 1,308
Pairs with MW divergence      : 1,055
Pairs dropped (always same MW): 253
Unique states in treated pairs: 49


## 5. Assign pair IDs and finalise

In [ ]:
# Assign a unique integer pair_id
diverging_pairs = diverging_pairs.reset_index(drop=True)
diverging_pairs.insert(0, "pair_id", diverging_pairs.index + 1)

# Rename for clarity
diverging_pairs = diverging_pairs.rename(
    columns={"state1_abbr": "state1", "state2_abbr": "state2"}
).drop(
    columns=["state1", "state2"], errors="ignore"
)  # drop numeric state cols

# Rebuild clean final columns
border_pairs = diverging_pairs[["pair_id", "fips1", "fips2"]].copy()
border_pairs["state1"] = border_pairs["fips1"].str[:2].map(fips_to_abbr)
border_pairs["state2"] = border_pairs["fips2"].str[:2].map(fips_to_abbr)

print("Final border_pairs shape:", border_pairs.shape)
print("Columns:", border_pairs.columns.tolist())
border_pairs.head(10)

Final border_pairs shape: (1055, 5)
Columns: ['pair_id', 'fips1', 'fips2', 'state1', 'state2']


,pair_id,fips1,fips2,state1,state2
0,1,01003,12033,AL,FL
1,2,01005,13061,AL,GA
2,3,01005,13239,AL,GA
3,4,01005,13259,AL,GA
4,5,01017,13145,AL,GA
5,6,01017,13285,AL,GA
6,7,01019,13055,AL,GA
7,8,01019,13115,AL,GA
8,9,01019,13233,AL,GA
9,10,01029,13045,AL,GA


## 6. Summary diagnostics

In [ ]:
print("=== Border pairs summary ===")
print(f"Total pairs          : {len(border_pairs):,}")
print(
    f"Unique counties      : {pd.concat([border_pairs['fips1'], border_pairs['fips2']]).nunique():,}"
)
print(
    f"Unique states        : {pd.concat([border_pairs['state1'], border_pairs['state2']]).nunique()}"
)

print("\nMost common state borders (top 10):")
state_pair_counts = (
    border_pairs.groupby(["state1", "state2"])
    .size()
    .reset_index(name="n_county_pairs")
    .sort_values("n_county_pairs", ascending=False)
)
print(state_pair_counts.head(10).to_string(index=False))

=== Border pairs summary ===
Total pairs          : 1,055
Unique counties      : 966
Unique states        : 49

Most common state borders (top 10):
state1 state2  n_county_pairs
    NC     VA              28
    VA     WV              27
    IL     MO              27
    AL     GA              27
    MI     WI              27
    KS     NE              26
    MN     WI              23
    NM     TX              23
    AR     MO              22
    IL     IN              22


In [ ]:
# Verify no duplicates
dupes = border_pairs.duplicated(subset=["fips1", "fips2"]).sum()
print(f"Duplicate pairs: {dupes}  (should be 0)")

# Verify pair_id is unique
print(
    f"Unique pair_ids: {border_pairs['pair_id'].nunique()} / {len(border_pairs)} (should match)"
)

# Missing values
print("\nMissing values:")
print(border_pairs.isnull().sum())

Duplicate pairs: 0  (should be 0)
Unique pair_ids: 1055 / 1055 (should match)

Missing values:
pair_id    0
fips1      0
fips2      0
state1     0
state2     0
dtype: int64


## 7. Save to parquet

In [12]:
border_pairs.to_parquet(OUTPUT, index=False)

print(f"Saved to  : {OUTPUT}")
print(f"File size : {OUTPUT.stat().st_size / 1e3:.1f} KB")
print(f"Rows      : {len(border_pairs):,}")

# Read back to verify
check = pd.read_parquet(OUTPUT)
print(f"\nRead back : {check.shape} — OK ✓")
check.head()

Saved to  : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/intermediate/border_pairs.parquet
File size : 17.0 KB
Rows      : 1,055

Read back : (1055, 5) — OK ✓


,pair_id,fips1,fips2,state1,state2
0,1,01003,12033,AL,FL
1,2,01005,13061,AL,GA
2,3,01005,13239,AL,GA
3,4,01005,13259,AL,GA
4,5,01017,13145,AL,GA


## 8. Findings & Notes

### Results

| Metric | Value |
|--------|-------|
| Total directed pairs parsed from adjacency file | 18,967 |
| Cross-state directed pairs | 2,616 |
| Unique cross-state pairs (deduplicated) | 1,308 |
| **Pairs with min wage divergence (kept)** | **1,055 (80.7%)** |
| Pairs dropped — always same wage | 253 (19.3%) |
| Unique border counties retained | 966 |
| Unique states represented | 49 / 51 |

### Most common state borders
| Border | County pairs |
|--------|-------------|
| NC – VA | 28 |
| VA – WV | 27 |
| IL – MO | 27 |
| AL – GA | 27 |
| MI – WI | 27 |
| KS – NE | 26 |
| MN – WI | 23 |
| NM – TX | 23 |
| AR – MO | 22 |
| IL – IN | 22 |

### Filtering logic
- **Cross-state only:** same-state adjacent pairs are dropped — they face the same minimum wage and provide no treatment variation.
- **Divergence required:** 253 pairs (19.3%) were dropped because both states always tracked the federal floor ($7.25). These pairs cannot contribute to the DiD estimate.

### Design rationale (Dube, Lester & Reich 2010)
Contiguous border counties share similar local labor markets, industry mix, and demand shocks. Any difference in employment trends within a pair is attributed to the minimum wage gap — not underlying economic differences. The 1,055 retained pairs across 966 unique counties and 49 states form the core sample for the causal analysis.